# Anime 101: Using Data to Find My First Anime
My partner is really into anime — like, really into it. Meanwhile, I’ve never watched a single episode. The anime scenes that pop up online sometimes look... questionable. So instead of jumping in blind, I decided to do what any good data analyst would do — grab all the data, analyse it, and find the perfect *first* anime for me.


In [27]:
import pandas as pd
anime = pd.read_csv('../data/anime.csv')
anime.head()

,Score,Popularity,Rank,Members,Description,Synonyms,Japanese,English,Type,Episodes,...,Premiered,Broadcast,Producers,Licensors,Studios,Source,Genres,Demographic,Duration,Rating
0,9.38,284,1,710,During their decade-long quest to defeat the D...,Frieren at the Funeral,葬送のフリーレン,Frieren: Beyond Journey's End,TV,28,...,Fall 2023,Fridays at 23:00 (JST),"Aniplex, Dentsu, Shogakukan-Shueisha Productio...","None found, add some",Madhouse,Manga,"AdventureAdventure, DramaDrama, FantasyFantasy",ShounenShounen,24 min. per ep.,PG-13 - Teens 13 or older
1,9.09,3,2,3,After a horrific alchemy experiment goes wrong...,"Hagane no Renkinjutsushi: Fullmetal Alchemist,...",鋼の錬金術師 FULLMETAL ALCHEMIST,Fullmetal Alchemist: Brotherhood,TV,64,...,Spring 2009,Sundays at 17:00 (JST),"Aniplex, Square Enix, Mainichi Broadcasting Sy...","Funimation, Aniplex of America",Bones,Manga,"ActionAction, AdventureAdventure, DramaDrama, ...",ShounenShounen,24 min. per ep.,R - 17+ (violence & profanity)
2,9.07,13,3,2,Eccentric scientist Rintarou Okabe has a never...,NaN,STEINS;GATE,Steins;Gate,TV,24,...,Spring 2011,Wednesdays at 02:05 (JST),"Frontier Works, Media Factory, Kadokawa Shoten...",Funimation,White Fox,Visual novel,"DramaDrama, Sci-FiSci-Fi, SuspenseSuspense",NaN,24 min. per ep.,PG-13 - Teens 13 or older
3,9.06,342,4,630,"Gintoki, Shinpachi, and Kagura return as the f...",Gintama' (2015),銀魂°,Gintama Season 4,TV,51,...,Spring 2015,Wednesdays at 18:00 (JST),"TV Tokyo, Aniplex, Dentsu","Funimation, Crunchyroll",Bandai Namco Pictures,Manga,"ActionAction, ComedyComedy, Sci-FiSci-Fi",ShounenShounen,24 min. per ep.,PG-13 - Teens 13 or older
4,9.05,21,5,2,Seeking to restore humanity's diminishing hope...,NaN,進撃の巨人 Season3 Part.2,Attack on Titan Season 3 Part 2,TV,10,...,Spring 2019,Mondays at 00:10 (JST),"Production I.G, Dentsu, Mainichi Broadcasting ...",Funimation,Wit Studio,Manga,"ActionAction, DramaDrama, SuspenseSuspense",ShounenShounen,23 min. per ep.,R - 17+ (violence & profanity)


# Data Cleanup
First things first, let's remove the animes that I would never watch like Specials

In [28]:
# Let's first get all of the genres avaiable

anime['GenreList'] = anime['Genres'].apply(lambda x: x.split(', ') if pd.notnull(x) else [])
all_genres = [genre for sublist in anime['GenreList'] for genre in sublist]
unique_genres = sorted(set(all_genres))
print(unique_genres)

['ActionAction', 'AdventureAdventure', 'Avant GardeAvant Garde', 'Award WinningAward Winning', 'Boys LoveBoys Love', 'ComedyComedy', 'DramaDrama', 'EcchiEcchi', 'FantasyFantasy', 'Girls LoveGirls Love', 'GourmetGourmet', 'HorrorHorror', 'MysteryMystery', 'RomanceRomance', 'Sci-FiSci-Fi', 'Slice of LifeSlice of Life', 'SportsSports', 'SupernaturalSupernatural', 'SuspenseSuspense']


Well, that's weird... Each genre seems to be written twice. Let's clean that up

In [29]:

def clean_genre_string(genre_str):
    if pd.isna(genre_str):
        return ''
    genres = genre_str.split(',')
    cleaned = []
    for g in genres:
        g = g.strip()
        half = len(g) // 2
        # If it's a doubled word like "DramaDrama", keep only the first half
        if len(g) % 2 == 0 and g[:half] == g[half:]:
            g = g[:half]
        cleaned.append(g)
    return ', '.join(cleaned)

anime['Genres'] = anime['Genres'].apply(clean_genre_string)
anime['GenreList'] = anime['Genres'].str.split(', ')
all_genres = [genre for sublist in anime['GenreList'] for genre in sublist]
unique_genres = sorted(set(all_genres))
print(unique_genres)

['', 'Action', 'Adventure', 'Avant Garde', 'Award Winning', 'Boys Love', 'Comedy', 'Drama', 'Ecchi', 'Fantasy', 'Girls Love', 'Gourmet', 'Horror', 'Mystery', 'Romance', 'Sci-Fi', 'Slice of Life', 'Sports', 'Supernatural', 'Suspense']


In [ ]:
# Fill missing ratings with 0 and genres with "Unknown"
anime['Score'] = anime['Score'].fillna(0)
anime['Genres'] = anime['Genres'].fillna('Unknown')
# Convert episodes to numeric, turn non-numeric into NaN
anime['Episodes'] = pd.to_numeric(anime['Episodes'], errors='coerce')

irrelevant_types = ['Avant Garde', 'Boys Love', 'Ecchi', 'Girls Love', 'Horror', 'Unknown', '']
anime = anime[anime['GenreList'].apply(lambda genres: not any(g in irrelevant_types for g in genres))]


GenreList
Action           352
Drama            263
Fantasy          247
Comedy           229
Adventure        214
Sci-Fi           170
Romance          151
Mystery          137
Supernatural     133
Award Winning     76
Suspense          65
Slice of Life     39
Sports            16
Gourmet            2
Name: count, dtype: int64


# Filtering Time
Data is now cleaned up and now it's time for me to filter the data to find the perfect next anime

In [36]:
starter_anime = anime[
    (anime['Episodes'] < 30) &
    (anime['Score'] > 8.0) &
    (anime['Type'] == 'TV')
]
starter_anime[['English', 'Genres', 'Episodes', 'Score']].sort_values(by='Score', ascending=False).head(10)


,English,Genres,Episodes,Score
0,Frieren: Beyond Journey's End,"Adventure, Drama, Fantasy",28.0,9.38
2,Steins;Gate,"Drama, Sci-Fi, Suspense",24.0,9.07
4,Attack on Titan Season 3 Part 2,"Action, Drama, Suspense",10.0,9.05
10,Gintama: Enchousen,"Action, Comedy, Sci-Fi",13.0,9.02
8,Bleach: Thousand-Year Blood War,"Action, Adventure, Fantasy",13.0,9.02
11,Kaguya-sama: Love is War - Ultra Romantic,"Comedy, Romance",13.0,9.01
12,Fruits Basket: The Final Season,"Drama, Romance, Supernatural",13.0,8.98
13,Gintama Season 5,"Action, Comedy, Sci-Fi",12.0,8.98
15,Clannad: After Story,"Drama, Romance, Supernatural",24.0,8.93
18,March Comes In Like a Lion 2nd Season,,22.0,8.92


## 🎯 So, What Anime Should I Watch First?

Based on the data, I’m leaning toward short, highly rated shows that aren’t too intense.  

Seems like a no brainer but to try this show called Frieren: Beyond Journey's End🫡

Wish me luck!